# RappiPlus: From Data to Business Decisions
### End-to-End Business Analytics Project — Python, SQL, Statistical Testing & BI Dashboarding

**Author:** David Anampa

This project evaluates the performance of **RappiPlus** to support data-driven business decisions, combining Python data cleaning, SQL analysis, statistical hypothesis testing, and a Power BI dashboard.

> 📊 **Live deliverable:** this project's final output includes an interactive Power BI dashboard. See [Section 6](#6.-Communicating-Results-—-Power-BI-Dashboard) for the link and a preview of what it covers.

## Project Overview

The goal of this project is to evaluate the performance of the RappiPlus service end-to-end, from raw data to an executive-ready dashboard.

The analysis combines multiple data sources:

- **rappiplus_orders_raw.csv** — orders, pricing, discounts, and revenue
- **rappiplus_catalog.csv** — product costs, categories, and suppliers
- **rappiplus_marketing_spend.csv** — marketing investment by channel and country
- **events / users / user_activity** (SQL database) — in-app user behavior
- **experiment_checkout_ui.csv** — results of an A/B experiment on the checkout flow

The workflow follows a clear, progressive logic:

1. 🔍 Assess whether the data can be trusted (data quality, Python)
2. 💰 Analyze whether the business is profitable (revenue, cost, and profit)
3. 🛒 Understand where users drop off (conversion funnel, SQL)
4. 🔁 Evaluate whether users come back (cohort retention, SQL)
5. 🧪 Validate whether a UI change made an impact (statistical test)
6. 📊 Communicate the results (BI dashboard)

Throughout the project, data is translated into insights that answer key business questions and support actionable recommendations.

---

## 1. Data Loading & Quality Validation

### 1.1 Loading & First Look

The three core business datasets are loaded and explored to understand their structure before any cleaning or analysis.

In [2]:
# importar librerías
# import libraries
import pandas as pd

In [2]:
# cargar archivos
# load files
orders = pd.read_csv('datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('datasets/rappiplus_marketing_spend.csv')

In [4]:
# explorar datasets
# explore datasets
def explore_dataset(name, df):
    print("=" * 80)
    print(f"DATASET: {name}")
    print("=" * 80)

    print(f"\nShape: {df.shape[0]} rows x {df.shape[1]} columns")

    print("\n--- First rows ---")
    display(df.head())

    print("\n--- General info (types and nulls) ---")
    df.info()

    print("\n--- % missing values per column ---")
    nulls = (df.isnull().sum() / len(df) * 100).round(2)
    print(nulls[nulls > 0].sort_values(ascending=False) if nulls.sum() > 0 else "No missing values")

    print("\n--- Fully duplicated rows ---")
    print(f"Exact duplicates: {df.duplicated().sum()}")

    print("\n--- Descriptive statistics (numeric) ---")
    display(df.describe())

    print("\n--- Descriptive statistics (categorical / text) ---")
    display(df.describe(include='object'))

    print("\n--- Unique values per column (top 10 columns with fewest uniques) ---")
    nunique = df.nunique().sort_values()
    print(nunique.head(10))

In [5]:
# tabla orders
# orders table
explore_dataset('orders', orders)

DATASET: orders

Shape: 25100 rows x 12 columns

--- First rows ---


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28



--- General info (types and nulls) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB

--- % missing values per column ---
pais                  1.20
categoria_producto    0.32
cantidad              

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06



--- Descriptive statistics (categorical / text) ---


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto
count,25100,25100,25100,24800,25080,25070,25070,25020
unique,25000,7642,181,6,2,3,7,3
top,order_3167,user_7769,2025-06-25,Colombia,desktop,social,Vacuum-Pro-Black,Hogar
freq,2,11,176,7520,12759,8428,4199,8385



--- Unique values per column (top 10 columns with fewest uniques) ---
dispositivo               2
fuente_referencia         3
categoria_producto        3
monto_descuento           4
pais                      6
cantidad                  6
nombre_producto           7
fecha_hora_pedido       181
id_usuario             7642
precio_unitario       19543
dtype: int64


In [6]:
# tabla catalog
# catalog table
explore_dataset('catalog', catalog)

DATASET: catalog

Shape: 7 rows x 4 columns

--- First rows ---


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"



--- General info (types and nulls) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes

--- % missing values per column ---
No missing values

--- Fully duplicated rows ---
Exact duplicates: 0

--- Descriptive statistics (numeric) ---


,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000



--- Descriptive statistics (categorical / text) ---


,nombre_producto,categoria_producto,proveedor
count,7,7,7
unique,7,3,7
top,Laptop-Gaming-16GB,Electrónica,"Fuller, Pena and Myers"
freq,1,3,1



--- Unique values per column (top 10 columns with fewest uniques) ---
categoria_producto    3
nombre_producto       7
costo_unitario        7
proveedor             7
dtype: int64


In [7]:
# tabla marketing
# marketing table
explore_dataset('marketing', marketing)

DATASET: marketing

Shape: 1620 rows x 5 columns

--- First rows ---


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40



--- General info (types and nulls) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB

--- % missing values per column ---
canal    6.23
dtype: float64

--- Fully duplicated rows ---
Exact duplicates: 0

--- Descriptive statistics (numeric) ---


,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000



--- Descriptive statistics (categorical / text) ---


,fecha,pais,id_campaña,canal
count,1620,1620,1620,1519
unique,180,3,9,3
top,2025-01-01,Mexico,organic_Mexico,paid_search
freq,9,540,180,507



--- Unique values per column (top 10 columns with fewest uniques) ---
pais             3
canal            3
id_campaña       9
fecha          180
gasto         1612
dtype: int64


---

### 1.2 Data Cleaning

Each of the three datasets is reviewed and cleaned to ensure the revenue, cost, and profitability analysis downstream is reliable:

- Validate and convert date columns to the correct format
- Review numeric variables (no negative or invalid zero values)
- Verify amount consistency
- Remove duplicates
- Review categorical variables

In [8]:
# fechas, duplicados y validación de montos/cantidades
# dates, duplicates, and amount/quantity validation
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')
orders = orders.drop_duplicates(subset='id_pedido').drop_duplicates()
orders = orders[(orders['cantidad'] > 0) & (orders['cantidad'] <= 100) & (orders['monto_total'] > 0)]
orders['pais'] = orders['pais'].str.strip().str.title()
orders = orders.dropna(subset=['id_usuario', 'monto_total'])

In [9]:
# normalizar texto
# normalize text
catalog['categoria_producto'] = catalog['categoria_producto'].str.strip().str.title()
catalog['nombre_producto'] = catalog['nombre_producto'].str.strip()

In [10]:
# fechas y nulos en canal
# dates and nulls in channel
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors='coerce')
marketing['canal'] = marketing['canal'].fillna('desconocido')
marketing['pais'] = marketing['pais'].str.strip().str.title()

In [11]:
# verificación rápida post-limpieza
# quick post-cleaning check
for name, df in {'orders': orders, 'catalog': catalog, 'marketing': marketing}.items():
    print(f"{name}: {df.shape[0]} rows | duplicates: {df.duplicated().sum()} | total nulls: {df.isnull().sum().sum()}")

print("\ncantidad range (orders):", orders['cantidad'].min(), "-", orders['cantidad'].max())

orders: 24936 rows | duplicates: 0 | total nulls: 406
catalog: 7 rows | duplicates: 0 | total nulls: 0
marketing: 1620 rows | duplicates: 0 | total nulls: 0

cantidad range (orders): 1.0 - 2.0


In [12]:
# nulos en orders
# missing values in orders
print(orders.isnull().sum()[orders.isnull().sum() > 0])

pais                  296
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     30
dtype: int64


In [13]:
# imputar 'pais' como 'desconocido', y eliminar el resto de nulos
# impute 'pais' as 'unknown', and drop the remaining nulls
orders['pais'] = orders['pais'].fillna('desconocido')
orders = orders.dropna(subset=['dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']).reset_index(drop=True)

print(f"orders final: {orders.shape[0]} rows | total nulls: {orders.isnull().sum().sum()}")

orders final: 24886 rows | total nulls: 0


**Diagnosis — data quality**

- `pais` had a small number of missing values that don't map to a clear customer segment, so they were imputed as `'desconocido'` (unknown) rather than dropped, to avoid losing otherwise valid orders.
- Missing values in `dispositivo`, `fuente_referencia`, `nombre_producto`, and `categoria_producto` were dropped, since these fields are not reliably imputable and are needed for downstream segmentation and joins.
- Invalid quantities (`cantidad` ≤ 0 or > 100) and non-positive amounts (`monto_total` ≤ 0) were removed as likely data entry errors.
- Exact duplicate rows and duplicate `id_pedido` values were removed to avoid double-counting revenue.

**Export:** once cleaning is finished, the three datasets are exported for use in the final stage of the project (the BI dashboard).

In [14]:
# exportar datasets
# export datasets
# Changed the separator to ";" due to issues with "," as a field separator
# delimiter ; and decimal ,
orders.to_csv('orders_clean.csv', index=False, encoding='utf-8-sig', sep=';', decimal=',')
catalog.to_csv('catalog_clean.csv', index=False, encoding='utf-8-sig', sep=';', decimal=',')
marketing.to_csv('marketing_clean.csv', index=False, encoding='utf-8-sig', sep=';', decimal=',')

---

## 2. Profitability & Sales KPIs

### 2.1 Core KPI Calculation

Using the three cleaned datasets (`orders`, `catalog`, `marketing`), the core business indicators are calculated to evaluate revenue, cost, and profitability, as well as sales behavior.

In [15]:
# revenue total
# total revenue
revenue_total = orders['monto_total'].sum()

In [16]:
# costo total (cruzando orders con catalog por nombre_producto)
# total cost (joining orders with catalog on nombre_producto)
orders_costo = orders.merge(catalog[['nombre_producto', 'costo_unitario']], on='nombre_producto', how='left')
costo_total = (orders_costo['cantidad'] * orders_costo['costo_unitario']).sum()

In [17]:
# inversión total en marketing
# total marketing investment
marketing_total = marketing['gasto'].sum()

In [18]:
# profit
profit = revenue_total - costo_total - marketing_total

In [19]:
print(f"Total revenue: ${revenue_total:,.2f}")
print(f"Total cost: ${costo_total:,.2f}")
print(f"Marketing invested: ${marketing_total:,.2f}")
print(f"Profit: ${profit:,.2f}")
print(f"Is it profitable? {'Yes' if profit > 0 else 'No'}")

Total revenue: $9,603,663.19
Total cost: $3,827,188.54
Marketing invested: $2,871,843.53
Profit: $2,904,631.12
Is it profitable? Yes


---

In [20]:

# ticket promedio, cantidad promedio, producto más vendido, gasto por canal
# average ticket, average quantity, top product, spend by channel
ticket_promedio = orders['monto_total'].mean()
cantidad_promedio = orders['cantidad'].mean()
producto_top = orders['nombre_producto'].value_counts().idxmax()
gasto_por_canal = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)

print(f"Average ticket: ${ticket_promedio:,.2f}")
print(f"Average quantity per order: {cantidad_promedio:.2f}")
print(f"Top-selling product: {producto_top}")
print("\nMarketing spend by channel:")
print(gasto_por_canal)

Average ticket: $385.91
Average quantity per order: 1.51
Top-selling product: Blender-XL-Red

Marketing spend by channel:
canal
social         918043.21
organic        913533.01
paid_search    863088.21
desconocido    177179.10
Name: gasto, dtype: float64


**Interpretation**

The revenue, cost, and marketing figures above determine whether RappiPlus is operating profitably once product costs and marketing investment are accounted for. The average ticket and quantity per order describe typical order size, the top-selling product highlights where demand concentrates, and the channel-level marketing spend shows where acquisition budget is currently allocated — a useful input for evaluating return on marketing investment in later stages.

## 3. Conversion Funnel Analysis (SQL)

### 3.1 Database Connection

User behavior inside the platform is analyzed directly from the production-like analytics database using SQL, starting with the **events** table.

> 🔒 **Note:** the original database credentials have been replaced with placeholders below. For a public portfolio, connection secrets should be provided through environment variables (e.g. a `.env` file loaded with `python-dotenv`) rather than hard-coded in the notebook.

In [2]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [3]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


### 3.2 Funnel Construction & Conversion Rates

In [4]:

# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT 
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY usuarios_unicos DESC;
'''
totals = pd.read_sql(query_totals, con=engine)
totals


,nombre_evento,usuarios_unicos
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT 
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios,
        CASE nombre_evento
            WHEN 'first_visit' THEN 1
            WHEN 'select_item' THEN 2
            WHEN 'add_to_cart' THEN 3
            WHEN 'begin_checkout' THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase' THEN 6
        END AS paso
    FROM events
    GROUP BY nombre_evento
)
SELECT 
    paso,
    nombre_evento,
    usuarios,
    LAG(usuarios) OVER (ORDER BY paso) AS usuarios_paso_anterior,
    ROUND(usuarios::numeric / LAG(usuarios) OVER (ORDER BY paso) * 100, 2) AS conversion_pct
FROM funnel
WHERE paso IS NOT NULL
ORDER BY paso;
'''
conversion = pd.read_sql(query_conversion, con=engine)
conversion

,paso,nombre_evento,usuarios,usuarios_paso_anterior,conversion_pct
0,1,first_visit,7796,NaN,NaN
1,2,select_item,7582,7796.0,97.26
2,3,add_to_cart,7634,7582.0,100.69
3,4,begin_checkout,7208,7634.0,94.42
4,5,add_payment_info,6250,7208.0,86.71
5,6,purchase,6240,6250.0,99.84


**Interpretation**

The funnel table above shows how many unique users reach each stage of the purchase journey — from first visit through to purchase — and the step-by-step conversion percentage between consecutive stages. The step with the lowest `conversion_pct` marks where the platform loses the largest share of users, and is the natural priority for UX or product improvements.

---

## 4. Cohort Retention Analysis (SQL)

### 4.1 Cohort Definition & Weekly Retention

Users are grouped into monthly cohorts based on their registration date, and weekly retention is calculated to understand whether users come back after registering. `retenido_w1`, `retenido_w2`, and `retenido_w3` represent users still active in week 1, 2, and 3 after registration, respectively.

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:

# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)


,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [ ]:

# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT 
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS mes_cohorte
    FROM users
),
retenidos AS (
    SELECT 
        c.mes_cohorte,
        COUNT(DISTINCT CASE WHEN a.dias_despues_registro = 7  AND a.activo = 1 THEN a.id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN a.dias_despues_registro = 14 AND a.activo = 1 THEN a.id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN a.dias_despues_registro = 21 AND a.activo = 1 THEN a.id_usuario END) AS retenido_w3
    FROM user_activity a
    JOIN cohortes c ON c.id_usuario = a.id_usuario
    GROUP BY c.mes_cohorte
),
tamano_cohorte AS (
    SELECT mes_cohorte, COUNT(DISTINCT id_usuario) AS clientes_iniciales
    FROM cohortes
    GROUP BY mes_cohorte
)
SELECT 
    t.mes_cohorte,
    t.clientes_iniciales,
    r.retenido_w1,
    r.retenido_w2,
    r.retenido_w3,
    ROUND(r.retenido_w1::numeric / t.clientes_iniciales * 100, 2) AS semana_1,
    ROUND(r.retenido_w2::numeric / t.clientes_iniciales * 100, 2) AS semana_2,
    ROUND(r.retenido_w3::numeric / t.clientes_iniciales * 100, 2) AS semana_3
FROM tamano_cohorte t
JOIN retenidos r ON t.mes_cohorte = r.mes_cohorte
ORDER BY t.mes_cohorte;
'''
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final


,mes_cohorte,clientes_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85


**Interpretation**

Comparing `semana_1`, `semana_2`, and `semana_3` across cohorts shows whether retention is improving, worsening, or stable over time, and whether the drop-off between week 1 and week 3 is steep (suggesting weak long-term engagement) or gradual (suggesting a healthier retention curve).

---

## 5. A/B Test — Checkout UI Experiment

### 5.1 Hypotheses & Test Selection

This experiment evaluates whether a change to the checkout UI impacts purchase conversion rate. The `convirtio` metric is 1 if the user completed the purchase and 0 otherwise.

**Hypotheses**
- **H₀:** there is no difference in conversion rate between the control group and the treatment group (new checkout UI); the conversion proportion is equal in both groups.
- **H₁:** there is a difference in conversion rate between the control group and the treatment group.

**Statistical test:** two-sample z-test for proportions, since `convirtio` is a binary (0/1) variable and two independent groups are being compared.
**Significance level (alpha):** 0.05

In [21]:
# exploramos la tabla
# explore the table
experiment = pd.read_csv('datasets/experiment_checkout_ui.csv')
experiment.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [22]:
from statsmodels.stats.proportion import proportions_ztest

# conversiones y tamaño de muestra por variante
# conversions and sample size per variant
resumen = experiment.groupby('variante')['convirtio'].agg(['sum', 'count'])
conversiones = resumen['sum'].values
n_usuarios = resumen['count'].values

z_stat, p_value = proportions_ztest(conversiones, n_usuarios)

print(resumen)
print(f"\nZ-stat: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

             sum  count
variante               
control      779   4965
tratamiento  820   5035

Z-stat: -0.8133
P-value: 0.4161


In [23]:
alpha = 0.05

if p_value < alpha:
    print(f"P-value ({p_value:.4f}) < alpha ({alpha}) -> reject H0")
    print("There is a statistically significant difference in conversion rate between variants.")
else:
    print(f"P-value ({p_value:.4f}) >= alpha ({alpha}) -> fail to reject H0")
    print("There is not enough evidence of a significant difference in conversion rate between variants.")

P-value (0.4161) >= alpha (0.05) -> fail to reject H0
There is not enough evidence of a significant difference in conversion rate between variants.


**Interpretation**

The z-test result above determines whether the new checkout UI produced a statistically significant change in conversion rate. As with the correlational analyses elsewhere in this portfolio, a significant result here shows an observed difference between groups — it does not, on its own, quantify the full business impact (e.g. revenue effect), which should be evaluated alongside the profitability KPIs from Section 2 before deciding to roll out the change.

---

## 6. Communicating Results — Power BI Dashboard

The three cleaned CSVs from Section 1 (`orders_clean.csv`, `catalog_clean.csv`, `marketing_clean.csv`) feed a Power BI dashboard built to communicate the sales, cost, marketing, and conversion results to stakeholders in a clear, visual way.

**Data preparation**
1. Load the cleaned CSVs into Power BI.
2. Build relationships: `orders.nombre_producto` → `catalog.nombre_producto`, and `orders.fecha_pedido` → a dedicated date table (to support YTD, YoY, and prior-period comparisons).
3. Create the calculated columns and measures needed for the visuals below.

**Dashboard 1 — Executive Overview**

*Key KPIs:* total revenue, total profit, total marketing spend, average ticket, average quantity per order.

*Visuals:* KPI cards for revenue, profit, and marketing spend; a line chart of monthly revenue/profit evolution and YTD trend; a bar chart of revenue and profit by product or category.

**Dashboard 2 — Detail / Drill-Through**

*Goal:* let stakeholders explore from the high-level KPIs down to individual orders or products.

*Visuals:* a detailed order table (product, quantity, revenue, cost, profit) with conditional formatting (negative profit in red, positive in green); a bar chart of units sold by product; drill-through from a selected product to all related orders; filters by date and product category.

### 📊 Dashboard Access

The full interactive dashboard (Executive Overview + Drill-through) was built in Power BI and is available here:

**🔗 [View the RappiPlus Dashboard](https://drive.google.com/drive/folders/1e7yHwJm8YKVyv2h-VxgedjDrMTvxgFXj?usp=drive_link)**

> If you're viewing this notebook on GitHub, dashboard screenshots can also be embedded directly below — add the image files to an `assets/` folder in the repository and reference them here, e.g. `![Executive Overview](assets/dashboard_overview.png)`.

---

# Executive Summary

## Objective
This project evaluates the end-to-end performance of RappiPlus — from data quality and profitability to user behavior, retention, and a checkout UI experiment — to support data-driven business decisions, with the results communicated through an interactive BI dashboard.

## Methodology
The workflow combined Python for data cleaning and KPI calculation, SQL for funnel and cohort retention analysis directly against the analytics database, a two-sample z-test for proportions to evaluate the checkout UI experiment, and Power BI for the final executive and drill-through dashboards.

## Key Deliverables
- Cleaned, validated datasets (`orders_clean.csv`, `catalog_clean.csv`, `marketing_clean.csv`) ready for reporting.
- Core profitability KPIs: revenue, cost, marketing investment, and profit.
- A SQL-based conversion funnel identifying where users drop off in the purchase journey.
- A SQL-based cohort retention analysis tracking user activity over the weeks following registration.
- A statistically grounded evaluation of the checkout UI experiment.
- An interactive Power BI dashboard (linked above) translating all of the above into an executive-ready view.

## Recommendations
- Prioritize the funnel stage with the steepest drop-off for UX or product investment.
- Track cohort retention over time to detect whether engagement is improving after any product changes.
- Combine the checkout UI test result with the profitability KPIs before deciding whether to roll out the new UI more broadly.
- Use the Power BI dashboard as the primary interface for stakeholders who need to monitor these metrics on an ongoing basis.